# Session 3, Live Demo: The Normal Distribution & the Central Limit Theorem

**Instructor demo notebook. Drive it conversationally, and make the room *predict before you compute*, three times today.**

Two legs: the **normal curve** (how unusual is one value) and the **Central Limit Theorem** (the one idea that makes testing a sample and trusting the whole population possible). The claim-amount and account-balance numbers are modelled on Ayushi's own Week 3 audit examples.

In [ ]:
# --- Generate the data (reproducible, fixed seed) ------------------
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(42)

# 1) Insurance claim amounts (EUR): modelled on Ayushi's Week 3 audit example
#    (her own numbers: mean EUR245.85, sd EUR36.33, n=100; we simulate more points
#    for a smoother picture, same mean and sd).
rng_c = np.random.default_rng(7)
CLAIM_MEAN, CLAIM_SD = 245.85, 36.33
claims = rng_c.normal(CLAIM_MEAN, CLAIM_SD, size=1000)

# 2) Returns and invoices: the SAME data Session 2 used (same seed), so the Q-Q plot
#    below is a callback, not new data.
returns_pct = rng.normal(8, 12, size=1000)
invoices = rng.lognormal(mean=7.6, sigma=0.8, size=1000)

print(f"claims:   mean {claims.mean():.2f}   sd {claims.std(ddof=1):.2f}")
print(f"returns:  mean {returns_pct.mean():.1f}%   sd {returns_pct.std():.1f}%")
print(f"invoices: mean EUR {invoices.mean():,.0f}   median EUR {np.median(invoices):,.0f}")

In [ ]:
# --- Load the room's own data ---------------------------------------
# The 'Countries Visited' Vevox poll. No internet, or the poll hasn't run yet?
# This falls back to a stand-in file so the demo still runs.
DATA_URL = ("https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/"
            "main/bu7852/session3/data/room_data.csv")
try:
    room = pd.read_csv(DATA_URL)
except Exception:
    try:
        from google.colab import files
        print("Upload room_data.csv ...")
        up = files.upload()
        room = pd.read_csv(next(iter(up)))
    except ImportError:
        room = pd.read_csv("room_data.csv")   # local fallback (course folder: data/room_data.csv)
room = room.iloc[:, 0].dropna().astype(int).to_numpy()
print(f"room: {len(room)} answers, mean {room.mean():.1f} countries, range {room.min()}-{room.max()}")

## Step 1, z-scores and the empirical rule: how unusual is one value?

**Predict before you compute:** a fund's annual returns average 8%, sd 12%. In what share of years does the return land within **one** standard deviation of average (-4% to 20%)? Write down a number, then run the cell.

The empirical rule is a property of the normal curve, so it only works on roughly-normal data. Fund returns qualify, and we check that with a Q-Q plot in Step 2. Claim amounts carry the CLT leg later.

In [ ]:
mu, sd = returns_pct.mean(), returns_pct.std(ddof=1)
within_1sd = np.mean(np.abs(returns_pct - mu) <= sd) * 100
within_2sd = np.mean(np.abs(returns_pct - mu) <= 2 * sd) * 100
print(f"within 1 sd: {within_1sd:.1f}%   (the empirical rule says ~68%)")
print(f"within 2 sd: {within_2sd:.1f}%   (the empirical rule says ~95%)")

plt.figure(figsize=(9, 4))
plt.hist(returns_pct, bins=30, density=True, color='#1F497D', alpha=0.45)
xs = np.linspace(returns_pct.min(), returns_pct.max(), 300)
plt.plot(xs, stats.norm.pdf(xs, mu, sd), color='#C0392B', lw=2.5, label='normal curve')
for k, c in [(1, '#3E8E5A'), (2, '#E8A33D')]:
    plt.axvspan(mu - k * sd, mu + k * sd, color=c, alpha=0.08)
plt.axvline(mu, color='#1F497D', ls='--', label='mean')
plt.xlabel('annual return (%)'); plt.legend(); plt.title('The empirical rule on fund returns')
plt.show()

# A -18% year: how unusual? (today's exercise in one line)
print(f"a -18% year: z = {(-18 - mu) / sd:.2f} SDs below average")

Now switch to claim amounts for an audit rule. **Claims beyond +/-2 SD are flagged for review**, about the furthest 5% on either side *when the amounts are roughly bell-shaped*. Check that shape with a Q-Q plot first; for skewed amounts, flag on percentiles instead. Cheap, defensible, and it scales from ten claims to a thousand (Ayushi's own audit rule).

In [ ]:
claim_mu, claim_sd = claims.mean(), claims.std(ddof=1)
z = (claims - claim_mu) / claim_sd
flagged = claims[np.abs(z) > 2]
print(f"{len(flagged)} of {len(claims)} claims flagged (|z| > 2)")
# Try one claim by hand:
example = 320
z_example = (example - claim_mu) / claim_sd
print(f"A EUR{example} claim has z = {z_example:.2f}")

## Step 2, the normality check: the Q-Q plot

One sentence to remember: **hug the line = normal, curve off = not normal.** Same returns/invoices data as Session 2's own displaying-data demo, a sharper test on data you've already seen.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
stats.probplot(returns_pct, dist='norm', plot=ax[0])
ax[0].set_title('Returns: hug the line = normal')
stats.probplot(invoices, dist='norm', plot=ax[1])
ax[1].set_title('Invoices: curve off = NOT normal')
plt.tight_layout(); plt.show()

**Claude beat:** describe one of these Q-Q plots to Claude and ask *"is this data normal, and how do you know?"* Check the answer against the picture, not the other way round.

## Step 3, the Central Limit Theorem, live from the room

**Predict before you compute:** if we plot everyone's 'countries visited', what shape will it be? Most rooms guess a bell. Run the cell and see.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(room, bins=range(1, room.max() + 2), color='#0D73B8', alpha=0.85)
plt.xlabel('countries visited'); plt.title('The room: not a bell')
plt.show()
print(f"Surprise 1: real data isn't normal. Skew, not symmetric.")

Now average small groups of the room's own answers, n = 2, 10, 30 at a time, and look at the **distribution of those averages**.

In [ ]:
def sample_means(pop, n, k=2000):
    return rng.choice(pop, size=(k, n), replace=True).mean(axis=1)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
ax[0].hist(room, bins=range(1, room.max() + 2), color='#0D73B8', alpha=0.85)
ax[0].set_title('Raw data (skewed)')
for a, n in zip(ax[1:], (2, 10, 30)):
    a.hist(sample_means(room, n), bins=30, color='#3E8E5A', alpha=0.85)
    a.set_title(f'Average of {n} = bell')
plt.tight_layout(); plt.show()
print("Surprise 2: averaging changed the shape. The bigger the sample, the tighter the bell.")

## Step 4, standard error: how fast the bell tightens

SE = sd / sqrt(n). **Predict before you compute:** sample 50 claims and average them. What range are you 95% sure that sample mean falls in? Write a number, then run the cell.

In [ ]:
n = 50
se = CLAIM_SD / np.sqrt(n)
lo, hi = CLAIM_MEAN - 2 * se, CLAIM_MEAN + 2 * se
print(f"SE = {CLAIM_SD}/sqrt({n}) = {se:.2f}")
print(f"95% range for the sample mean: EUR{lo:.2f} to EUR{hi:.2f}")

# Prove it by simulation: draw 4,000 samples of n=50 claims, average each one.
sample_means_50 = rng.choice(claims, size=(4000, n), replace=True).mean(axis=1)
plt.figure(figsize=(8, 4))
plt.hist(sample_means_50, bins=40, color='#0D73B8', alpha=0.85)
plt.axvspan(lo, hi, color='#E8A33D', alpha=0.2)
plt.axvline(CLAIM_MEAN, color='#C0392B', label='population mean')
plt.xlabel('sample mean (EUR)'); plt.legend()
plt.title('4,000 samples of n=50 claims, each averaged'); plt.show()

**Three conditions make this work** (guess them before you look): the observations must be **independent**, **identically distributed**, and have **finite variance**. Almost always true in business data; worth naming, not worth losing sleep over.

## Step 5, a preview, not a lesson: next week's confidence interval

Ayushi's own audit example. We are not deriving the formula today, just looking at what it produces: the sample mean, plus and minus a multiple of its standard error, same move as Step 4, just dressed up with a name next week.

In [ ]:
BAL_MEAN, BAL_SD, BAL_N = 1445.92, 288.88, 50
se_bal = BAL_SD / np.sqrt(BAL_N)
# With a sample this size, next week's method uses the t-distribution, not z, for the
# multiplier. Shown here only to match the number on the slide; the why is next week's.
t_mult = stats.t.ppf(0.975, BAL_N - 1)
ci = (BAL_MEAN - t_mult * se_bal, BAL_MEAN + t_mult * se_bal)
print(f"95% CI for the account balance: EUR{ci[0]:,.2f} to EUR{ci[1]:,.2f}")

recorded = 1600
verdict = 'OUTSIDE the interval, flag it' if not ci[0] <= recorded <= ci[1] else 'inside, consistent with the sample'
print(f"A recorded balance of EUR{recorded}: {verdict}")

## Wrap, why today matters

A z-score makes one value legible. The CLT is why testing a **sample** licenses a claim about the whole **population**, the mathematical engine under audit sampling and the Deloitte capstone.

**Next week (Session 4):** confidence intervals and hypothesis testing, built directly on the CLT you just met.

Habit check: did you **predict before you computed**, three times today?